# Mangrove carbon and coastal avoided-EAD multicriteria analysis

This notebook ranks Forces of Nature mangrove patches using their estimated carbon benefit and their coastal-flood avoided expected annual damages (EADs). The aim is to test how protection priorities change when carbon and risk-reduction benefits are weighted differently.

The main criteria are total patch carbon, in tonnes C, and patch-level avoided coastal-flood EAD, in US$/year. Scores are percentile based so that very large outliers do not dominate the combined score.

In [ ]:
import os
import sys
from pathlib import Path

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
matplotlib_config_dir = base_path / ".matplotlib"
matplotlib_config_dir.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(matplotlib_config_dir))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.cm import ScalarMappable
from matplotlib.colors import ListedColormap, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter

robyn_libraries_path = (base_path / "robyns_libraries").resolve()
if str(robyn_libraries_path) not in sys.path:
    sys.path.append(str(robyn_libraries_path))
import Robyn_paper_2_defs

pd.set_option("display.max_columns", 200)

## Configuration

In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"
priority_threshold_quantile = 0.75
top_priority_count = 20

mangrove_carbon_path = base_path / "dphil_paper_3/processed_data/carbon/global_carbon_benefit/fn_mangrove_patch_global_carbon_benefit_summary.gpkg"
jamaica_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg"
administrative_boundaries_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/admin_boundaries.gpkg"

output_dir = base_path / "dphil_paper_3/processed_data/carbon/global_carbon_benefit_mangrove_mca"
figure_dir = base_path / "dphil_paper_3/results/co_benefits/carbon/global_carbon_benefit_mangrove_mca"
output_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

mca_summary_csv_path = output_dir / "fn_mangrove_carbon_coastal_ead_mca_summary.csv"
mca_summary_gpkg_path = output_dir / "fn_mangrove_carbon_coastal_ead_mca_summary.gpkg"
scenario_summary_csv_path = output_dir / "fn_mangrove_carbon_coastal_ead_mca_scenario_summary.csv"
top_priority_csv_path = output_dir / "fn_mangrove_carbon_coastal_ead_mca_top_priority_patches.csv"
sensitivity_csv_path = output_dir / "fn_mangrove_carbon_coastal_ead_mca_sensitivity.csv"

scatter_plot_path = figure_dir / "fn_mangrove_carbon_vs_coastal_avoided_ead_scatter.png"
weight_scenario_map_path = figure_dir / "fn_mangrove_carbon_coastal_ead_mca_weight_scenarios_map.png"
quadrant_map_path = figure_dir / "fn_mangrove_carbon_coastal_ead_priority_quadrants_map.png"
rank_sensitivity_plot_path = figure_dir / "fn_mangrove_carbon_coastal_ead_mca_top20_sensitivity.png"

required_paths = [mangrove_carbon_path, jamaica_boundary_path, administrative_boundaries_path]
for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required input: {required_path}")

weight_scenarios = pd.DataFrame(
    [
        {"weight_scenario": "equal_priority", "ead_weight": 0.50, "carbon_weight": 0.50, "label": "Equal priority"},
        {"weight_scenario": "risk_reduction_priority", "ead_weight": 0.75, "carbon_weight": 0.25, "label": "Risk-reduction priority"},
        {"weight_scenario": "carbon_priority", "ead_weight": 0.25, "carbon_weight": 0.75, "label": "Carbon priority"},
    ]
)
ead_cases = pd.DataFrame(
    [
        {"ead_case": "min", "ead_column": "avoided_ead_usd_min", "label": "Minimum avoided EAD"},
        {"ead_case": "mean", "ead_column": "avoided_ead_usd_mean", "label": "Mean avoided EAD"},
        {"ead_case": "max", "ead_column": "avoided_ead_usd_max", "label": "Maximum avoided EAD"},
    ]
)

print(f"Mangrove carbon/EAD input: {mangrove_carbon_path}")
print(f"Outputs: {output_dir}")
display(weight_scenarios)
display(ead_cases)

## Load mangrove carbon and avoided-EAD data

In [ ]:
mangrove_patches = gpd.read_file(mangrove_carbon_path).to_crs(jamaica_metric_grid_crs)
jamaica_boundary = gpd.read_file(jamaica_boundary_path).to_crs(jamaica_metric_grid_crs)
administrative_boundaries = gpd.read_file(administrative_boundaries_path, layer="admin1").to_crs(jamaica_metric_grid_crs)

mangrove_patches = mangrove_patches[mangrove_patches.geometry.notna() & ~mangrove_patches.geometry.is_empty].copy()
jamaica_boundary = jamaica_boundary[jamaica_boundary.geometry.notna() & ~jamaica_boundary.geometry.is_empty].copy()
administrative_boundaries = administrative_boundaries[
    administrative_boundaries.geometry.notna() & ~administrative_boundaries.geometry.is_empty
].copy()

required_mangrove_columns = [
    "Mangrove_ID",
    "Parish",
    "patch_area_ha",
    "total_carbon_benefit_with_nn_fill",
    "carbon_benefit_per_patch_ha_with_nn_fill",
    "avoided_ead_usd_min",
    "avoided_ead_usd_max",
    "patch_positive_avoided_EADs",
]
missing_mangrove_columns = [column for column in required_mangrove_columns if column not in mangrove_patches.columns]
if missing_mangrove_columns:
    raise KeyError(f"Missing required mangrove columns: {missing_mangrove_columns}")

mangrove_mca = mangrove_patches.copy()
mangrove_mca["carbon_total_tonnes_c"] = pd.to_numeric(
    mangrove_mca["total_carbon_benefit_with_nn_fill"], errors="coerce"
).fillna(0)
mangrove_mca["carbon_density_tonnes_c_per_ha"] = pd.to_numeric(
    mangrove_mca["carbon_benefit_per_patch_ha_with_nn_fill"], errors="coerce"
).fillna(0)
mangrove_mca["avoided_ead_usd_min"] = pd.to_numeric(mangrove_mca["avoided_ead_usd_min"], errors="coerce").fillna(0)
mangrove_mca["avoided_ead_usd_max"] = pd.to_numeric(mangrove_mca["avoided_ead_usd_max"], errors="coerce").fillna(0)
mangrove_mca["avoided_ead_usd_mean"] = pd.to_numeric(
    mangrove_mca["patch_positive_avoided_EADs"], errors="coerce"
).fillna(0)
mangrove_mca["patch_area_ha"] = pd.to_numeric(mangrove_mca["patch_area_ha"], errors="coerce").fillna(0)
mangrove_mca["carbon_tonnes_c_per_ha"] = np.where(
    mangrove_mca["patch_area_ha"] > 0,
    mangrove_mca["carbon_total_tonnes_c"] / mangrove_mca["patch_area_ha"],
    np.nan,
)
mangrove_mca["avoided_ead_usd_per_ha_mean"] = np.where(
    mangrove_mca["patch_area_ha"] > 0,
    mangrove_mca["avoided_ead_usd_mean"] / mangrove_mca["patch_area_ha"],
    np.nan,
)

input_summary = pd.DataFrame(
    [
        {"metric": "mangrove_patches", "value": len(mangrove_mca)},
        {"metric": "patch_area_ha", "value": mangrove_mca["patch_area_ha"].sum()},
        {"metric": "total_carbon_tonnes_c", "value": mangrove_mca["carbon_total_tonnes_c"].sum()},
        {"metric": "avoided_ead_usd_min", "value": mangrove_mca["avoided_ead_usd_min"].sum()},
        {"metric": "avoided_ead_usd_mean", "value": mangrove_mca["avoided_ead_usd_mean"].sum()},
        {"metric": "avoided_ead_usd_max", "value": mangrove_mca["avoided_ead_usd_max"].sum()},
    ]
)
display(input_summary)
display(
    mangrove_mca[
        [
            "Mangrove_ID",
            "Parish",
            "patch_area_ha",
            "carbon_total_tonnes_c",
            "carbon_density_tonnes_c_per_ha",
            "avoided_ead_usd_mean",
        ]
    ].head()
)

## Score carbon and avoided-EAD criteria

The MCA uses percentile scores, where the highest patch for a criterion receives a value close to 1 and the lowest receives a value close to 0. This avoids letting a single very high avoided-EAD patch dominate the entire ranking. Rank-sum fields are also retained to match the simpler paper-2 MCA style.

In [ ]:
def percentile_score(values):
    numeric_values = pd.to_numeric(values, errors="coerce").fillna(0)
    if numeric_values.nunique(dropna=False) <= 1:
        return pd.Series(1.0, index=numeric_values.index)
    return numeric_values.rank(method="average", pct=True)


def descending_rank(values):
    numeric_values = pd.to_numeric(values, errors="coerce").fillna(0)
    return numeric_values.rank(ascending=False, method="min").astype(int)


mangrove_mca["carbon_score"] = percentile_score(mangrove_mca["carbon_total_tonnes_c"])
mangrove_mca["carbon_rank"] = descending_rank(mangrove_mca["carbon_total_tonnes_c"])

for ead_case in ead_cases.itertuples(index=False):
    ead_score_column = f"ead_score_{ead_case.ead_case}"
    ead_rank_column = f"ead_rank_{ead_case.ead_case}"
    rank_sum_column = f"rank_sum_{ead_case.ead_case}"
    mangrove_mca[ead_score_column] = percentile_score(mangrove_mca[ead_case.ead_column])
    mangrove_mca[ead_rank_column] = descending_rank(mangrove_mca[ead_case.ead_column])
    mangrove_mca[rank_sum_column] = mangrove_mca["carbon_rank"] + mangrove_mca[ead_rank_column]

    for weight_scenario in weight_scenarios.itertuples(index=False):
        score_column = f"mca_score_{ead_case.ead_case}_{weight_scenario.weight_scenario}"
        rank_column = f"mca_rank_{ead_case.ead_case}_{weight_scenario.weight_scenario}"
        mangrove_mca[score_column] = (
            weight_scenario.ead_weight * mangrove_mca[ead_score_column]
            + weight_scenario.carbon_weight * mangrove_mca["carbon_score"]
        )
        mangrove_mca[rank_column] = descending_rank(mangrove_mca[score_column])

carbon_priority_threshold = mangrove_mca["carbon_total_tonnes_c"].quantile(priority_threshold_quantile)
ead_priority_threshold = mangrove_mca["avoided_ead_usd_mean"].quantile(priority_threshold_quantile)
mangrove_mca["high_carbon_total"] = mangrove_mca["carbon_total_tonnes_c"] >= carbon_priority_threshold
mangrove_mca["high_avoided_ead_mean"] = mangrove_mca["avoided_ead_usd_mean"] >= ead_priority_threshold
mangrove_mca["priority_quadrant"] = np.select(
    [
        mangrove_mca["high_carbon_total"] & mangrove_mca["high_avoided_ead_mean"],
        mangrove_mca["high_avoided_ead_mean"] & ~mangrove_mca["high_carbon_total"],
        mangrove_mca["high_carbon_total"] & ~mangrove_mca["high_avoided_ead_mean"],
    ],
    ["High EAD + high carbon", "High EAD only", "High carbon only"],
    default="Lower EAD + lower carbon",
)

display(
    mangrove_mca.sort_values("mca_rank_mean_equal_priority")[
        [
            "Mangrove_ID",
            "Parish",
            "patch_area_ha",
            "carbon_total_tonnes_c",
            "avoided_ead_usd_mean",
            "carbon_rank",
            "ead_rank_mean",
            "mca_score_mean_equal_priority",
            "mca_rank_mean_equal_priority",
            "priority_quadrant",
        ]
    ].head(top_priority_count)
)

## Summarise ranking sensitivity

In [ ]:
scenario_summary_rows = []
for ead_case in ead_cases.itertuples(index=False):
    for weight_scenario in weight_scenarios.itertuples(index=False):
        score_column = f"mca_score_{ead_case.ead_case}_{weight_scenario.weight_scenario}"
        rank_column = f"mca_rank_{ead_case.ead_case}_{weight_scenario.weight_scenario}"
        top_priority_patches = mangrove_mca[mangrove_mca[rank_column] <= top_priority_count]
        scenario_summary_rows.append(
            {
                "ead_case": ead_case.ead_case,
                "weight_scenario": weight_scenario.weight_scenario,
                "ead_weight": weight_scenario.ead_weight,
                "carbon_weight": weight_scenario.carbon_weight,
                "top_patch_count": len(top_priority_patches),
                "top_patch_area_ha": top_priority_patches["patch_area_ha"].sum(),
                "top_carbon_tonnes_c": top_priority_patches["carbon_total_tonnes_c"].sum(),
                "top_avoided_ead_usd_min": top_priority_patches["avoided_ead_usd_min"].sum(),
                "top_avoided_ead_usd_mean": top_priority_patches["avoided_ead_usd_mean"].sum(),
                "top_avoided_ead_usd_max": top_priority_patches["avoided_ead_usd_max"].sum(),
                "top_carbon_share": top_priority_patches["carbon_total_tonnes_c"].sum()
                / mangrove_mca["carbon_total_tonnes_c"].sum(),
                "top_mean_ead_share": top_priority_patches["avoided_ead_usd_mean"].sum()
                / mangrove_mca["avoided_ead_usd_mean"].sum(),
                "best_patch_id": int(mangrove_mca.sort_values(rank_column).iloc[0]["Mangrove_ID"]),
            }
        )
scenario_summary = pd.DataFrame(scenario_summary_rows)

scenario_rank_columns = [column for column in mangrove_mca.columns if column.startswith("mca_rank_")]
sensitivity_rows = []
for mangrove_patch in mangrove_mca.itertuples(index=False):
    patch_ranks = pd.Series({column: getattr(mangrove_patch, column) for column in scenario_rank_columns})
    sensitivity_rows.append(
        {
            "Mangrove_ID": int(mangrove_patch.Mangrove_ID),
            "Parish": mangrove_patch.Parish,
            "carbon_total_tonnes_c": mangrove_patch.carbon_total_tonnes_c,
            "avoided_ead_usd_mean": mangrove_patch.avoided_ead_usd_mean,
            "best_mca_rank": int(patch_ranks.min()),
            "worst_mca_rank": int(patch_ranks.max()),
            "mean_mca_rank": float(patch_ranks.mean()),
            "top20_scenario_count": int((patch_ranks <= top_priority_count).sum()),
            "top20_all_scenarios": bool((patch_ranks <= top_priority_count).all()),
        }
    )
sensitivity_summary = pd.DataFrame(sensitivity_rows).sort_values(
    ["top20_scenario_count", "mean_mca_rank"], ascending=[False, True]
)

top_priority_table = mangrove_mca.sort_values("mca_rank_mean_equal_priority")[
    [
        "Mangrove_ID",
        "Parish",
        "TYPE",
        "patch_area_ha",
        "carbon_total_tonnes_c",
        "carbon_density_tonnes_c_per_ha",
        "avoided_ead_usd_min",
        "avoided_ead_usd_mean",
        "avoided_ead_usd_max",
        "carbon_rank",
        "ead_rank_mean",
        "rank_sum_mean",
        "mca_score_mean_equal_priority",
        "mca_rank_mean_equal_priority",
        "mca_rank_mean_risk_reduction_priority",
        "mca_rank_mean_carbon_priority",
        "priority_quadrant",
    ]
].head(top_priority_count)

display(scenario_summary)
display(sensitivity_summary.head(top_priority_count))
display(top_priority_table)

## Relationship between carbon and avoided EAD

In [ ]:
relationship_data = mangrove_mca[
    ["carbon_total_tonnes_c", "carbon_density_tonnes_c_per_ha", "avoided_ead_usd_mean", "avoided_ead_usd_min", "avoided_ead_usd_max"]
].replace([np.inf, -np.inf], np.nan)
relationship_summary = pd.DataFrame(
    [
        {
            "relationship": "total_carbon_vs_mean_avoided_ead",
            "pearson": relationship_data["carbon_total_tonnes_c"].corr(
                relationship_data["avoided_ead_usd_mean"], method="pearson"
            ),
            "spearman": relationship_data["carbon_total_tonnes_c"].corr(
                relationship_data["avoided_ead_usd_mean"], method="spearman"
            ),
        },
        {
            "relationship": "carbon_density_vs_mean_avoided_ead",
            "pearson": relationship_data["carbon_density_tonnes_c_per_ha"].corr(
                relationship_data["avoided_ead_usd_mean"], method="pearson"
            ),
            "spearman": relationship_data["carbon_density_tonnes_c_per_ha"].corr(
                relationship_data["avoided_ead_usd_mean"], method="spearman"
            ),
        },
    ]
)
display(relationship_summary)

## Save tables

In [ ]:
mca_summary_columns = [column for column in mangrove_mca.columns if column != "geometry"]
mangrove_mca[mca_summary_columns].to_csv(mca_summary_csv_path, index=False)
mangrove_mca.to_file(mca_summary_gpkg_path, driver="GPKG")
scenario_summary.to_csv(scenario_summary_csv_path, index=False)
top_priority_table.to_csv(top_priority_csv_path, index=False)
sensitivity_summary.to_csv(sensitivity_csv_path, index=False)

print(f"Saved MCA patch CSV: {mca_summary_csv_path}")
print(f"Saved MCA patch GeoPackage: {mca_summary_gpkg_path}")
print(f"Saved scenario summary: {scenario_summary_csv_path}")
print(f"Saved top-priority table: {top_priority_csv_path}")
print(f"Saved sensitivity table: {sensitivity_csv_path}")

## Figures

In [ ]:
score_colour_map = plt.get_cmap("viridis")
score_norm = Normalize(vmin=0, vmax=1)
quadrant_order = [
    "High EAD + high carbon",
    "High EAD only",
    "High carbon only",
    "Lower EAD + lower carbon",
]
quadrant_colours = {
    "High EAD + high carbon": "#1B9E77",
    "High EAD only": "#D95F02",
    "High carbon only": "#7570B3",
    "Lower EAD + lower carbon": "#BDBDBD",
}
quadrant_colour_map = ListedColormap([quadrant_colours[quadrant] for quadrant in quadrant_order])
administrative_boundary_handle = Line2D(
    [0], [0], color="#424242", linewidth=0.7, label="Administrative boundaries"
)


def style_jamaica_map(axis, title_text):
    axis.set_title(title_text, pad=6)
    axis.set_axis_off()
    axis.set_aspect("equal")
    Robyn_paper_2_defs.draw_scale_bar(
        axis,
        location=(0.88, 0.78),
        length_km=20,
        linewidth=0.6,
        label_offset=0.02,
        km_offset=0.01,
    )
    Robyn_paper_2_defs.draw_north_arrow(axis, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)


def plot_map_base(axis):
    jamaica_boundary.boundary.plot(ax=axis, color="#757575", linewidth=0.45)
    administrative_boundaries.boundary.plot(ax=axis, color="#424242", linewidth=0.25)


def add_score_colourbar(figure, axes, label_text):
    scalar_mappable = ScalarMappable(norm=score_norm, cmap=score_colour_map)
    scalar_mappable.set_array([])
    colourbar = figure.colorbar(
        scalar_mappable,
        ax=axes,
        orientation="horizontal",
        fraction=0.035,
        pad=0.025,
        shrink=0.72,
    )
    colourbar.set_label(label_text, fontsize=9)
    colourbar.ax.tick_params(labelsize=8)
    return colourbar


def format_tonnes_axis(value, position):
    if value >= 1_000_000:
        return f"{value / 1_000_000:g}m"
    if value >= 1_000:
        return f"{value / 1_000:g}k"
    return f"{value:g}"


def format_usd_axis(value, position):
    if value >= 1_000_000:
        return f"${value / 1_000_000:g}m"
    if value >= 1_000:
        return f"${value / 1_000:g}k"
    return f"${value:g}"

In [ ]:
figure, axes = plt.subplots(3, 1, figsize=(8.27, 11.69), constrained_layout=True)
map_scenarios = [
    ("mca_score_mean_equal_priority", "a) Equal carbon and avoided-EAD priority"),
    ("mca_score_mean_risk_reduction_priority", "b) Risk-reduction priority"),
    ("mca_score_mean_carbon_priority", "c) Carbon priority"),
]
for axis, (score_column, title_text) in zip(axes, map_scenarios):
    mangrove_mca.plot(
        ax=axis,
        column=score_column,
        cmap=score_colour_map,
        vmin=0,
        vmax=1,
        linewidth=0.35,
        edgecolor="black",
        missing_kwds={"color": "lightgrey", "label": "No score"},
    )
    plot_map_base(axis)
    style_jamaica_map(axis, title_text)
add_score_colourbar(figure, axes, "Mangrove priority score, percentile weighted")
figure.savefig(weight_scenario_map_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved weight-scenario map: {weight_scenario_map_path}")

In [ ]:
quadrant_code_lookup = {quadrant: code for code, quadrant in enumerate(quadrant_order)}
mangrove_mca["priority_quadrant_code"] = mangrove_mca["priority_quadrant"].map(quadrant_code_lookup)

figure, axis = plt.subplots(figsize=(10, 6))
mangrove_mca.plot(
    ax=axis,
    column="priority_quadrant_code",
    categorical=True,
    cmap=quadrant_colour_map,
    linewidth=0.35,
    edgecolor="black",
)
plot_map_base(axis)
style_jamaica_map(axis, "Mangrove carbon and avoided-EAD priority quadrants")
quadrant_handles = [Patch(facecolor=quadrant_colours[quadrant], edgecolor="black", label=quadrant) for quadrant in quadrant_order]
axis.legend(handles=quadrant_handles + [administrative_boundary_handle], loc="lower left", frameon=True, fontsize=8)
figure.tight_layout()
figure.savefig(quadrant_map_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved quadrant map: {quadrant_map_path}")

In [ ]:
scatter_data = mangrove_mca[
    (mangrove_mca["carbon_total_tonnes_c"] > 0) & (mangrove_mca["avoided_ead_usd_mean"] > 0)
].copy()
scatter_colours = scatter_data["priority_quadrant"].map(quadrant_colours)

figure, axis = plt.subplots(figsize=(8, 6))
axis.scatter(
    scatter_data["carbon_total_tonnes_c"],
    scatter_data["avoided_ead_usd_mean"],
    s=np.clip(scatter_data["patch_area_ha"] / 8, 12, 180),
    c=scatter_colours,
    edgecolor="black",
    linewidth=0.25,
    alpha=0.82,
)
axis.set_xscale("log")
axis.set_yscale("log")
axis.xaxis.set_major_formatter(FuncFormatter(format_tonnes_axis))
axis.yaxis.set_major_formatter(FuncFormatter(format_usd_axis))
axis.set_xlabel("Total patch carbon (tonnes C, log scale)")
axis.set_ylabel("Mean avoided coastal-flood EAD (US$/year, log scale)")
axis.set_title("Mangrove carbon and coastal avoided EAD")
axis.grid(True, which="both", linestyle=":", linewidth=0.4, alpha=0.55)

labelled_patches = mangrove_mca.sort_values("mca_rank_mean_equal_priority").head(8)
for labelled_patch in labelled_patches.itertuples(index=False):
    if labelled_patch.carbon_total_tonnes_c > 0 and labelled_patch.avoided_ead_usd_mean > 0:
        axis.annotate(
            str(int(labelled_patch.Mangrove_ID)),
            xy=(labelled_patch.carbon_total_tonnes_c, labelled_patch.avoided_ead_usd_mean),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=8,
        )

scatter_handles = [Patch(facecolor=quadrant_colours[quadrant], edgecolor="black", label=quadrant) for quadrant in quadrant_order]
axis.legend(handles=scatter_handles, loc="lower right", frameon=True, fontsize=8)
figure.tight_layout()
figure.savefig(scatter_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved scatter plot: {scatter_plot_path}")

In [ ]:
top_sensitivity_plot_data = sensitivity_summary.head(top_priority_count).sort_values("top20_scenario_count")
figure, axis = plt.subplots(figsize=(9, 6))
axis.barh(
    top_sensitivity_plot_data["Mangrove_ID"].astype(str),
    top_sensitivity_plot_data["top20_scenario_count"],
    color="#2E7D32",
    edgecolor="black",
    linewidth=0.35,
)
axis.set_xlabel("Number of MCA scenarios where patch is in top 20")
axis.set_ylabel("Mangrove patch ID")
axis.set_title("Sensitivity of top mangrove priorities across EAD and weight scenarios")
axis.set_xlim(0, len(ead_cases) * len(weight_scenarios))
axis.grid(axis="x", linestyle=":", linewidth=0.4, alpha=0.6)
figure.tight_layout()
figure.savefig(rank_sensitivity_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved rank-sensitivity plot: {rank_sensitivity_plot_path}")

## How to use these outputs

- Use `mca_rank_mean_equal_priority` as the main balanced carbon/coastal-risk prioritisation.
- Use `mca_rank_mean_risk_reduction_priority` where avoided coastal EAD is the primary decision criterion.
- Use `mca_rank_mean_carbon_priority` where carbon is the primary decision criterion.
- Use the minimum and maximum EAD rank fields for uncertainty/sensitivity checks.
- Interpret the scores as a relative ranking across the modelled mangrove patches, not as an absolute valuation of carbon and avoided damages.